# 02 — Clustering and PCA (T-011/S4)

Input: production (`full` text) e5-small catalog embeddings from notebook 01. Components verified: spherical k-means (themes, multi-interest profiles, map colouring), its HDBSCAN comparator, and PCA (2-D product map + dimensionality study). See `docs/AI_MVP_DESIGN.md` §6 and §8.

**Pre-declared k rule:** among k ∈ {10, 15, …, 60}, discard k with mean pairwise seed-ARI < 0.7 (unstable), then pick the k with the best average rank over cosine silhouette (↑), Davies–Bouldin (↓) and NMI against top-level OFF categories (↑). Internal scores alone are not treated as semantic truth.

In [1]:
import json, hashlib, platform, random, time, itertools
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score as ARI, normalized_mutual_info_score as NMI
from sklearn.cluster import HDBSCAN
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import os, sys; os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from ml.asap_ml import retrieval as R, structure as S
OUT = ROOT / 'notebooks/results/02_clustering_pca'; OUT.mkdir(parents=True, exist_ok=True)
SEED = 20261006; random.seed(SEED); np.random.seed(SEED)
cat = R.load_catalog(); X = np.load('data/processed/embeddings_e5-small.npy')
nb01 = json.loads(Path('notebooks/results/01_data_embeddings_retrieval/summary.json').read_text())
assert hashlib.sha256(X.tobytes()).hexdigest() == nb01['embeddings_sha256'] and len(cat) == len(X)
texts = [R.product_text(p) for p in cat]
top = np.array([p['top_category'] for p in cat]); leaf = np.array([p['category'] for p in cat])
env = {'python': platform.python_version(), 'sklearn': sklearn.__version__, 'numpy': np.__version__,
       'run_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
X.shape, env

((10000, 384),
 {'python': '3.13.0',
  'sklearn': '1.9.1',
  'numpy': '2.5.3',
  'run_utc': '2026-10-06T13:02:02Z'})

## 1. Representation comparison and k sweep
A first run on the full-text embedding (A) produced language/country clusters rather than product types (low purity, unstable). Three representations are therefore compared under the same rule: **A** full-text e5 (retrieval space); **B** e5 of the English OFF taxonomy text only (product-type space); **C** A with its top 5 principal components removed (all-but-the-top debiasing). Caveat: B is built from category text, so its agreement with categories is partly by construction; stability and silhouette are the independent evidence, and the purpose of themes is precisely product-type grouping.

In [2]:
type_texts = [R.product_text(p, 'type') for p in cat]
XB, _ = R.encode('e5-small', type_texts)
mA, cA, _ = S.pca_fit(X); XC = R.normalize((X - mA) - ((X - mA) @ cA[:5].T) @ cA[:5])
REPS = {'A_full': X, 'B_type': XB, 'C_debiased': XC}
ks = list(range(10, 61, 5)); seeds = [SEED + i for i in range(5)]
def k_sweep(V):
    rows = []
    for k in ks:
        t = time.perf_counter(); runs = [S.spherical_kmeans(V, k, s)[0] for s in seeds]; fit_s = (time.perf_counter() - t) / len(seeds)
        lab = runs[0]
        rows.append({'k': k, 'silhouette': silhouette_score(V, lab, metric='cosine', sample_size=5000, random_state=SEED),
                     'davies_bouldin': davies_bouldin_score(V, lab), 'stability_ari': np.mean([ARI(a, b) for a, b in itertools.combinations(runs, 2)]),
                     'nmi_top': NMI(top, lab), 'nmi_leaf': NMI(leaf, lab), 'fit_s': fit_s})
    return pd.DataFrame(rows).set_index('k')
sweeps = {r: k_sweep(V) for r, V in REPS.items()}
pd.concat(sweeps, axis=1).round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

A_full                                                          B_type  \
   silhouette davies_bouldin stability_ari nmi_top nmi_leaf  fit_s silhouette   
k                                                                               
10      0.053          4.778         0.684   0.320    0.345  0.410      0.123   
15      0.052          4.474         0.731   0.388    0.411  0.448      0.156   
20      0.049          4.463         0.651   0.418    0.452  0.471      0.156   
25      0.052          4.287         0.647   0.438    0.474  0.576      0.175   
30      0.054          4.413         0.694   0.445    0.499  0.688      0.181   
35      0.061          4.311         0.617   0.492    0.531  0.775      0.206   
40      0.052          4.182         0.630   0.482    0.538  0.860      0.222   
45      0.058          4.219         0.595   0.500    0.553  1.010      0.228   
50      0.056          4.018         0.587   0.517    0.575  1.176      0.257   
55      0.059          4.122         0.576   0.541    0.590  1.402      0.261   
60      0.063          4.033         0.540   0.537    0.602  1.314      0.279   

                                                        C_debiased  \
   davies_bouldin stability_ari nmi_top nmi_leaf  fit_s silhouette   
k                                                                    
10          3.391         0.555   0.533    0.470  0.269      0.033   
15          3.171         0.617   0.597    0.549  0.351      0.040   
20          2.945         0.554   0.644    0.589  0.432      0.044   
25          2.842         0.532   0.635    0.631  0.529      0.048   
30          2.640         0.560   0.660    0.636  0.606      0.053   
35          2.480         0.541   0.667    0.648  0.720      0.055   
40          2.427         0.544   0.682    0.682  0.775      0.062   
45          2.302         0.551   0.699    0.694  0.893      0.060   
50          2.254         0.594   0.700    0.709  0.995      0.060   
55          2.323         0.610   0.701    0.721  1.089      0.063   
60          2.204         0.618   0.724    0.726  1.179      0.065   

                                                         
   davies_bouldin stability_ari nmi_top nmi_leaf  fit_s  
k                                                        
10          6.126         0.459   0.222    0.310  0.344  
15          5.438         0.576   0.314    0.393  0.515  
20          5.174         0.601   0.322    0.420  0.564  
25          4.951         0.577   0.359    0.458  0.625  
30          4.831         0.563   0.413    0.501  0.748  
35          4.593         0.567   0.436    0.522  0.828  
40          4.470         0.578   0.456    0.553  0.920  
45          4.594         0.565   0.473    0.566  1.051  
50          4.419         0.577   0.485    0.584  1.097  
55          4.204         0.571   0.501    0.596  1.237  
60          4.271         0.610   0.510    0.608  1.314

In [3]:
def choose_k(sw):
    st = sw[sw.stability_ari >= 0.7]
    if st.empty: return None
    rk = st.silhouette.rank(ascending=False) + st.davies_bouldin.rank() + st.nmi_top.rank(ascending=False)
    return int(rk.idxmin())
best = {r: choose_k(sw) for r, sw in sweeps.items()}
rep_table = pd.DataFrame({r: {'k': best[r], **(sweeps[r].loc[best[r]].to_dict() if best[r] else {})} for r in REPS}).T
best_original = best[rep_table.dropna(subset=['k']).sort_values(['stability_ari', 'silhouette'], ascending=False).index[0]]
original_choice = rep_table.dropna(subset=['k']).sort_values(['stability_ari', 'silhouette'], ascending=False).index[0]
print('original pre-declared rule selects:', original_choice, 'k =', best[original_choice]); rep_table.round(4)

original pre-declared rule selects: A_full k = 15


,k,silhouette,davies_bouldin,stability_ari,nmi_top,nmi_leaf,fit_s
A_full,15.0,0.0525,4.4735,0.7311,0.3879,0.4108,0.4484
B_type,NaN,NaN,NaN,NaN,NaN,NaN,NaN
C_debiased,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Rule revision (user decision, 2026-10-06, D-031)
The original rule selects A_full at k = 15, whose clusters split products by **language/country** rather than product type (see the c-TF-IDF terms of the first run). B_type is better on every quality metric (silhouette 3–5× higher, lower Davies–Bouldin, higher NMI) but no representation except A reaches seed stability 0.7, even with `n_init = 20` (B ≈ 0.58–0.68). Themes exist to group product types for the user, so the revised rule selects **B_type** and chooses k ∈ {20, …, 60} by the best average rank of silhouette (↑), Davies–Bouldin (↓), NMI vs top-level categories (↑) and stability (↑). The moderate stability is reported as a limitation; the app uses one fixed, versioned clustering. B's category agreement is partly by construction.

In [4]:
REP = 'B_type'; Xc = REPS[REP]; sweep = sweeps[REP]
cand = sweep.loc[20:60]
rk = cand.silhouette.rank(ascending=False) + cand.davies_bouldin.rank() + cand.nmi_top.rank(ascending=False) + cand.stability_ari.rank(ascending=False)
best[REP] = int(rk.idxmin()); rk.rename('rank_sum').to_frame().T

k,20,25,30,35,40,45,50,55,60
rank_sum,31.0,34.0,25.0,26.0,22.0,17.0,11.0,10.0,4.0


In [5]:
K = best[REP]; print(REP, '| chosen k =', K, '| stability at K =', round(sweep.loc[K, 'stability_ari'], 3))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
sweep[['silhouette']].plot(ax=ax[0], marker='o', title='Cosine silhouette (5k sample)'); sweep[['stability_ari']].plot(ax=ax[1], marker='o', title='Seed stability (mean ARI)')
sweep[['nmi_top', 'nmi_leaf']].plot(ax=ax[2], marker='o', title='NMI vs OFF categories')
for a in ax: a.axvline(K, ls='--', c='grey')
plt.tight_layout(); plt.savefig(OUT / 'k_sweep.png', dpi=150); plt.show()

B_type | chosen k = 60 | stability at K = 0.618


/tmp/ipykernel_223850/152372166.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'k_sweep.png', dpi=150); plt.show()


## 2. Comparator: HDBSCAN on PCA-50
Density-based clustering does not need k but labels low-density points as noise; the app needs every product in a theme.

In [6]:
mean, comps, evr = S.pca_fit(X)  # retrieval space (dimensionality study)
mc, cc, evr_c = S.pca_fit(Xc)  # clustering space (HDBSCAN input, map)
Z50 = S.pca_project(Xc, mc, cc, 50)
t = time.perf_counter()
hd = HDBSCAN(min_cluster_size=25, min_samples=5).fit_predict(Z50); hd_s = time.perf_counter() - t
mask = hd >= 0
labels_k, centroids = S.spherical_kmeans(Xc, K, SEED, n_init=20)
comparison = pd.DataFrame({
  f'kmeans_k{K}': {'clusters': K, 'noise_share': 0.0, 'nmi_top': NMI(top, labels_k), 'nmi_top_on_hdbscan_core': NMI(top[mask], labels_k[mask])},
  'hdbscan': {'clusters': int(hd.max() + 1), 'noise_share': float(1 - mask.mean()), 'nmi_top': NMI(top, hd), 'nmi_top_on_hdbscan_core': NMI(top[mask], hd[mask])}}).T
comparison['fit_s'] = [sweep.loc[K, 'fit_s'], hd_s]; comparison.round(4)

/home/mih/cs/elfak/master/is/asap-app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,clusters,noise_share,nmi_top,nmi_top_on_hdbscan_core,fit_s
kmeans_k60,60.0,0.0000,0.7240,0.8013,1.1789
hdbscan,132.0,0.3132,0.6298,0.7838,2.9148


## 3. Cluster themes
Automatic label = dominant leaf category in the cluster; c-TF-IDF terms are shown for review. Labels are machine-generated and still need a human pass before they are shown in the UI as final.

In [7]:
themes = S.label_clusters(labels_k, cat, texts)
th = pd.DataFrame(themes).set_index('cluster')
pd.set_option('display.max_rows', 80, 'display.max_colwidth', 70)
th.sort_values('size', ascending=False)

,size,label,label_share,top_category,top_purity,terms
cluster,,,,,,
12,479,Mashed vegetables,0.021,plant-based-foods-and-beverages,0.547,"[juices, fruit, nectars, jams, concentrate]"
2,380,Salami,0.150,meats-and-their-products,0.924,"[meats, sausages, prepared, salami, hams]"
8,329,Cheeses,0.182,dairies,0.793,"[milks, cheeses, fermented, cow, uht]"
0,304,Milk chocolates,0.095,snacks,0.770,"[chocolates, chocolate, cocoa, dark, filled]"
14,303,Sweetened beverages,0.432,beverages-and-beverages-preparations,0.541,"[beverages, sweetened, sodas, cola, carbonated]"
40,297,Canned tunas,0.108,seafood,0.906,"[fishes, tunas, canned, salmons, fillets]"
26,296,Dog food,0.368,pet-food,0.595,"[dog, pet, dry, jack, wet]"
7,288,Biscuits,0.247,snacks,0.990,"[biscuits, cakes, crackers, wafers, appetizers]"
25,287,Rolled oats,0.080,plant-based-foods-and-beverages,0.941,"[cereals, flakes, breads, rolled, cereal]"


In [8]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
th['size'].sort_values().plot.barh(ax=ax[0], title=f'Cluster sizes (k={K})', fontsize=6)
ax[1].hist(th.top_purity, bins=10, range=(0, 1)); ax[1].set_title('Top-level category purity per cluster')
plt.tight_layout(); plt.savefig(OUT / 'clusters.png', dpi=150); plt.show()
purity = {'median_top_purity': float(th.top_purity.median()), 'median_label_share': float(th.label_share.median()),
          'smallest': int(th['size'].min()), 'largest': int(th['size'].max())}; purity

/tmp/ipykernel_223850/1909176834.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'clusters.png', dpi=150); plt.show()


{'median_top_purity': 0.7755000000000001,
 'median_label_share': 0.3235,
 'smallest': 36,
 'largest': 479}

## 4. Degenerate cases
Assignment of a new (uncatalogued) product by nearest centroid must be deterministic, finite and agree with the fit.

In [9]:
agree = float((S.assign(Xc, centroids) == labels_k).mean())
zero = S.assign(R.normalize(np.zeros((1, Xc.shape[1]))), centroids)
dup = S.assign(np.vstack([Xc[0], Xc[0]]), centroids)
edge = {'refit_assign_agreement': agree, 'zero_vector_assigns_to': int(zero[0]), 'duplicates_same_cluster': bool(dup[0] == dup[1]),
        'empty_clusters': int(K - len(np.unique(labels_k)))}; edge

{'refit_assign_agreement': 1.0,
 'zero_vector_assigns_to': 0,
 'duplicates_same_cluster': True,
 'empty_clusters': 0}

## 5. PCA: explained variance

In [10]:
cum = np.cumsum(evr)
var_summary = {f'pc{d}': float(cum[d - 1]) for d in (2, 16, 32, 64, 128)}
var_summary.update({'dims_for_80pct': int(np.searchsorted(cum, 0.8) + 1), 'dims_for_95pct': int(np.searchsorted(cum, 0.95) + 1)})
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.plot(np.arange(1, len(cum) + 1), cum); ax.set_xlabel('components'); ax.set_ylabel('cumulative explained variance')
ax.axhline(0.8, ls=':', c='grey'); ax.axhline(0.95, ls=':', c='grey'); ax.set_title('PCA on e5-small catalog embeddings')
plt.tight_layout(); plt.savefig(OUT / 'pca_variance.png', dpi=150); plt.show(); var_summary

/tmp/ipykernel_223850/298189855.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'pca_variance.png', dpi=150); plt.show(); var_summary


{'pc2': 0.058329302817583084,
 'pc16': 0.2511441111564636,
 'pc32': 0.36571091413497925,
 'pc64': 0.5170230269432068,
 'pc128': 0.7134570479393005,
 'dims_for_80pct': 169,
 'dims_for_95pct': 281}

## 6. PCA: does reduction preserve retrieval?
Overlap@10 with full-dimension neighbours and full-text nDCG@10 (category-inflated, used only for the *relative* loss). Rule from the design: adopt a reduced d only if overlap ≥ 0.95 and nDCG@10 loss ≤ 0.01.

In [11]:
from collections import defaultdict
groups_by = defaultdict(set)
for i, p in enumerate(cat): groups_by[p['category']].add(i)
groups = {i: groups_by[p['category']] for i, p in enumerate(cat)}
queries = sorted(random.Random(SEED).sample([i for i in range(len(cat)) if len(groups[i]) >= 5], 500))
full_top = {q: set(R.top_k(X, q, 10)) for q in queries}; base = R.evaluate(X, queries, groups)['ndcg@10']
red = []
for d in (16, 32, 64, 128):
    Xd = R.normalize(S.pca_project(X, mean, comps, d))
    ov = np.mean([len(full_top[q] & set(R.top_k(Xd, q, 10))) / 10 for q in queries])
    nd = R.evaluate(Xd, queries, groups)['ndcg@10']
    red.append({'d': d, 'overlap@10': ov, 'ndcg@10': nd, 'ndcg_loss': base - nd, 'adopt': ov >= 0.95 and base - nd <= 0.01})
red = pd.DataFrame(red).set_index('d'); print('full-dimension nDCG@10 =', round(base, 4)); red.round(4)

full-dimension nDCG@10 = 0.4796


,overlap@10,ndcg@10,ndcg_loss,adopt
d,,,,
16,0.2954,0.2849,0.1946,False
32,0.4526,0.3853,0.0943,False
64,0.6006,0.4503,0.0293,False
128,0.7362,0.4761,0.0035,False


## 7. 2-D product map
PCA(2) is a linear projection, so clusters overlap; it is an orientation aid, not a quality claim. Colours = 12 largest clusters.

In [12]:
Z2 = S.pca_project(Xc, mc, cc, 2); cum_c = np.cumsum(evr_c)
big = th['size'].sort_values(ascending=False).index[:12]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(Z2[:, 0], Z2[:, 1], s=2, c='lightgrey')
for c in big:
    m = labels_k == c; ax.scatter(Z2[m, 0], Z2[m, 1], s=3, label=f"{th.loc[c, 'label'][:22]} ({m.sum()})")
ax.legend(fontsize=7, markerscale=4, loc='best'); ax.set_title(f'PCA(2) map ({REP}), PC1+PC2 = {cum_c[1]:.1%} variance')
plt.tight_layout(); plt.savefig(OUT / 'pca_map.png', dpi=150); plt.show()
sep = {'nmi_top_vs_kmeans_on_2d': NMI(labels_k, S.assign(R.normalize(Z2), R.normalize(np.vstack([Z2[labels_k == c].mean(0) for c in range(K)]))))}; sep

/tmp/ipykernel_223850/2061815079.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'pca_map.png', dpi=150); plt.show()


{'nmi_top_vs_kmeans_on_2d': 0.44831502156750924}

## 8. Runtime (desktop CPU)

In [13]:
q = Xc[123:124]
rt = {'assign_single': R.timed(lambda: S.assign(q, centroids), 500), 'pca2_project_single': R.timed(lambda: S.pca_project(q, mc, cc, 2), 500),
      'kmeans_fit_k': {'mean_s': float(sweep.loc[K, 'fit_s'])}, 'hdbscan_fit': {'s': hd_s}}
rt

{'assign_single': {'median_ms': 0.006823000148870051,
  'p95_ms': 0.009321545076090839},
 'pca2_project_single': {'median_ms': 0.0043479994928929955,
  'p95_ms': 0.004589454329106957},
 'kmeans_fit_k': {'mean_s': 1.1789168092000182},
 'hdbscan_fit': {'s': 2.914762188003806}}

## 9. Export (git-ignored runtime artifacts + committed results)

In [14]:
P = ROOT / 'data/processed'
np.save(P / 'cluster_centroids.npy', centroids); np.save(P / 'cluster_labels.npy', labels_k.astype(np.int32))
np.savez(P / 'pca.npz', mean=mc, components=cc[:2]); np.save(P / f'embeddings_cluster_{REP}.npy', Xc); (P / 'cluster_themes.json').write_text(json.dumps(themes, ensure_ascii=False, indent=1))
rng = np.random.default_rng(SEED); sample = np.sort(np.concatenate([rng.choice(np.flatnonzero(labels_k == c), min(25, (labels_k == c).sum()), replace=False) for c in range(K)]))
(P / 'map_sample.json').write_text(json.dumps([{'id': cat[i]['id'], 'x': round(float(Z2[i, 0]), 5), 'y': round(float(Z2[i, 1]), 5), 'cluster': int(labels_k[i])} for i in sample]))
summary = {'env': env, 'k_sweep': {str(k): {c: round(float(v), 4) for c, v in r.items()} for k, r in sweep.iterrows()}, 'k': K, 'representation': REP, 'original_rule_choice': {'representation': original_choice, 'k': best_original}, 'representation_table': {r: {c: (None if pd.isna(v) else round(float(v), 4)) for c, v in row.items()} for r, row in rep_table.iterrows()},
  'comparison': {i: {c: round(float(v), 4) for c, v in r.items()} for i, r in comparison.iterrows()}, 'purity': purity, 'edge_cases': edge,
  'pca_variance': var_summary, 'pca_reduction': {str(d): {c: (bool(v) if c == 'adopt' else round(float(v), 4)) for c, v in r.items()} for d, r in red.iterrows()},
  'full_dim_ndcg@10': base, 'map_sample_size': int(len(sample)), 'map_2d_separation': sep, 'runtime_ms': rt,
  'artifact_sha256': {f: hashlib.sha256((P / f).read_bytes()).hexdigest() for f in ('cluster_centroids.npy', 'cluster_labels.npy', 'pca.npz')}}
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2, default=float)); sweep.round(4).to_csv(OUT / 'k_sweep.csv')
th.to_csv(OUT / 'cluster_themes.csv'); red.round(4).to_csv(OUT / 'pca_reduction.csv')
print(json.dumps({k: summary[k] for k in ('k', 'purity', 'pca_variance', 'edge_cases')}, indent=1))

{
 "k": 60,
 "purity": {
  "median_top_purity": 0.7755000000000001,
  "median_label_share": 0.3235,
  "smallest": 36,
  "largest": 479
 },
 "pca_variance": {
  "pc2": 0.058329302817583084,
  "pc16": 0.2511441111564636,
  "pc32": 0.36571091413497925,
  "pc64": 0.5170230269432068,
  "pc128": 0.7134570479393005,
  "dims_for_80pct": 169,
  "dims_for_95pct": 281
 },
 "edge_cases": {
  "refit_assign_agreement": 1.0,
  "zero_vector_assigns_to": 0,
  "duplicates_same_cluster": true,
  "empty_clusters": 0
 }
}


## Limitations
- OFF categories are crowd-sourced and imperfect; NMI against them is an external sanity check, not ground truth.
- Silhouette is computed on a seeded 5,000-point sample.
- Theme labels are automatic; a human review pass is pending before UI use.
- PCA(2) preserves little global variance of 384-d embeddings; non-linear maps (UMAP/t-SNE) would separate better but are not required.